In [ ]:
from pathlib import Path
import scanpy as sc
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from pprint import pprint

sc.set_figure_params(frameon=False, dpi=100, fontsize=12, dpi_save=300)
plt.rcParams["svg.fonttype"] = "none"

import scatlastb_utils as atl
from batch_analysis_utils import plot_diff_per_clusters

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
figure_dir = Path('figures/figure4')
figure_dir.mkdir(exist_ok=True, parents=True)

## Read data

In [ ]:
adata = atl.io.read_anndata(
    'data/output/batch_analysis/hlca_epithelial.zarr',
    dask=True,
    backed=True,
)

In [ ]:
integration1 = 'scanvi_dataset'
integration2 = 'scanvi_sample'

In [ ]:
adata.obs['ann_level_5'] = adata.obs['ann_level_5'].cat.remove_categories(["None"])

In [ ]:
for color in [
    'ann_finest_level',
    'ann_level_5',
    # f'majority_reference--{integration1}',
    # f'majority_reference--{integration2}',
    'anatomical_region_ccf_score',
    'tissue_level_2',
]:
    kwargs = dict(
        color=color,
        size=3,
        palette=sc.pl.palettes.default_102,
        # plot_centroids=[color] if color.startswith('ann') else None,
        # max_label_length=2,
        # verbose=False,
    )
    sc.pl.embedding(adata, basis=f'X_umap--{integration1}', **kwargs)
    sc.pl.embedding(adata, basis=f'X_umap--{integration2}', **kwargs)

## Visualise globally computed graph dissim

In [ ]:
comparisons = [
    f'{integration1}-vs-{integration2}',
    f'{integration2}-vs-{integration1}'
]

metrics = ['avg_distance_diff']

In [ ]:
for _metric in metrics:
    gd_colors = [f'{_metric}:{comparison}' for comparison in comparisons]

    for col in gd_colors:
        adata.obs[f'{col}_bool'] = adata.obs[f'{col}'] > 0.1
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=gd_colors,
        cmap='viridis',
        size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[gd_colors].quantile(0.999).tolist(),
        sort_order=True,
        # mask_obs=mask,
    )
 
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=gd_colors,
        cmap='viridis',
        size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[gd_colors].quantile(0.999).tolist(),
        sort_order=True,
        # mask_obs=mask,
    )

## Compute graph dissimilarity

In [ ]:
%%time
comp = f'{integration1}-vs-{integration2}'

kwargs = dict(
    scale_distances=True,
    log_scale_diffs=False,
    # k_max=adata.uns['neighbors']['params']['n_neighbors'],
    n_jobs=5,
)

adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration1}',
    obsp_distances_1=f'distances--{integration1}',
    obsm_key_1=f'X_emb--{integration1}',
    obsp_connectivities_2=f'connectivities--{integration2}',
    obsp_distances_2=f'distances--{integration2}',
    obsm_key_2=f'X_emb--{integration2}',
    **kwargs,
)

comp = f'{integration2}-vs-{integration1}'
adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = atl.metrics.compare_distances(
    adata,
    obsp_connectivities_1=f'connectivities--{integration2}',
    obsp_distances_1=f'distances--{integration2}',
    obsm_key_1=f'X_emb--{integration2}',
    obsp_connectivities_2=f'connectivities--{integration1}',
    obsp_distances_2=f'distances--{integration1}',
    obsm_key_2=f'X_emb--{integration1}',
    **kwargs,
)

In [ ]:
atl.me.plot_ranked_distances(adata, integration1, integration2)
atl.me.plot_distances_scatter(adata, integration1, integration2, s=1, alpha=0.5)
atl.me.plot_distances_scatter(adata, integration2, integration1, s=1, alpha=0.5)

In [ ]:
comparisons = [
    f'{integration1}-vs-{integration2}',
    f'{integration2}-vs-{integration1}'
]

metrics = ['avg_distance_diff']

In [ ]:
for _metric in metrics:
    gd_colors = [f'{_metric}:{comparison}' for comparison in comparisons]

    kwargs = dict(
        color=gd_colors,
        cmap='viridis',
        size=10,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax='p99.9',
        sort_order=True,
    )
    sc.pl.embedding(adata, basis=f'X_umap--{integration1}', **kwargs)
    sc.pl.embedding(adata, basis=f'X_umap--{integration2}', **kwargs)

# Cell type-level evaluation

In [ ]:
comparison = comparisons[1]
metric = 'avg_distance_diff'

In [ ]:
# compare batch=dataset --> batch=sample
plot_diff_per_clusters(
    adata,
    'ann_level_5',
    f'{metric}:{comparisons[0]}',
    # cluster_thresh=10,
    # quantile=0.99,
    agg='median',
)

In [ ]:
# compare batch=sample --> batch=dataset
plot_diff_per_clusters(
    adata,
    'ann_level_5',
    f'{metric}:{comparisons[1]}',
    # cluster_thresh=10,
    # quantile=0.99,
    agg='median',
)

In [ ]:
# compare batch=sample --> batch=dataset
ann_clusters = plot_diff_per_clusters(
    adata,
    'ann_finest_level',
    f'{metric}:{comparisons[1]}',
    # cluster_thresh=10,
    # quantile=0.99,
    agg='median',
    width=5,
    height_factor=0.4,
)
ann_clusters

## Compare sample --> dataset

### Fig. 4A

In [ ]:
comparison = comparisons[1]

In [ ]:
# compare batch=sample --> batch=dataset
ann_clusters = plot_diff_per_clusters(
    adata,
    'ann_finest_level',
    f'{metric}:{comparison}',
    # cluster_thresh=10,
    # quantile=0.99,
    agg='median',
    width=4,
    height_factor=0.3,
    save=figure_dir / '4A_graph_dissim.svg',
)

In [ ]:
n_top = 3

groups = ann_clusters.sort_values(ascending=False).head(n_top).index.tolist()
mask = adata.obs['ann_finest_level'].isin(groups+['Multiciliated (non-nasal)'])

adata.obs['ann_finest_level_select'] = adata.obs.loc[mask, 'ann_finest_level'].cat.remove_unused_categories()

In [ ]:
for col in [
    'ann_finest_level',
    # f'leiden_0.5_1--{integration1}',
    # f'leiden_0.5_1--{integration2}',
]:
    kwargs = dict(
        verbose=False,
        color=col,
        plot_centroids=[col],
        max_label_length=2,
        mask_obs=mask,
        size=5,
        adjust_text_kwargs = {
            "arrowprops": {"arrowstyle": "-", "color": "black", "lw": 1},
            "expand_text": (1.05, 1.2),
            "expand_points": (2.0, 2.0), # ← expand bbox vs points; bigger = more gap from clusters
            "force_points": (1.5, 1.0), # ← repulsion strength fr%%!om points; bigger = pushed further
        }
    )
    atl.pl.embedding(adata, basis=f'X_umap--{integration1}', **kwargs)
    atl.pl.embedding(adata, basis=f'X_umap--{integration2}', **kwargs)

In [ ]:
for color in [
    'ann_finest_level',
    'ann_level_5',
]:
    kwargs = dict(
        color=color,
        # plot_centroids=[color],
        max_label_length=2,
        size=2,
        verbose=False,
        output_dir=figure_dir,
        file_type='svg',
        adjust_text_kwargs = {
            "arrowprops": {"arrowstyle": "-", "lw": 1},
            "expand_text": (1.05, 1.2),
            "expand_points": (2.0, 2.0), # ← expand bbox vs points; bigger = more gap from clusters
            "force_points": (1.5, 1.0), # ← repulsion strength from points; bigger = pushed further
        }
        
    )
    atl.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        file_prefix=f'4B_{integration1}_',
        **kwargs,
    )
    atl.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        file_prefix=f'4B_{integration2}_',
        **kwargs,
    )

### Fig. 4B

In [ ]:
metrics = [
    # 'avg_distance_1',
    # 'avg_distance_2',
    # 'avg_difference',
    'avg_distance_diff',
    # 'spearman_correlation'
]

In [ ]:
for _metric in metrics:
    gd_colors = [f'{_metric}:{comparison}'] # for comparison in comparisons]
    
    fig = sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=gd_colors,
        cmap='viridis',
        size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax='p99.9',
        sort_order=True,
        mask_obs=mask,
        return_fig=True,
    )
    fig.savefig(figure_dir / f'Fig4B_{integration1}--{metric}.svg')
 
    fig = sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=gd_colors,
        cmap='viridis',
        size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax='p99.9',
        sort_order=True,
        mask_obs=mask,
        return_fig=True,
    )
    fig.savefig(figure_dir / f'Fig4B_{integration2}--{metric}.svg')

## Compare dataset --> sample

In [ ]:
comparison = comparisons[0]

In [ ]:
# compare batch=sample --> batch=dataset
ann_clusters = plot_diff_per_clusters(
    adata,
    'ann_finest_level',
    f'{metric}:{comparisons[0]}',
    # cluster_thresh=10,
    # quantile=0.99,
    agg='median',
    width=5,
    height_factor=0.4,
)
# plt.savefig(figure_dir / 'graph_dissim.svg')

In [ ]:
n_top = 5

groups = ann_clusters.sort_values(ascending=False).head(n_top).index.tolist()
mask = adata.obs['ann_finest_level'].isin(groups)

In [ ]:
for col in [
    'ann_finest_level',
    f'leiden_0.5_1--{integration1}',
    f'leiden_0.5_1--{integration2}',
]:
    kwargs = dict(
        verbose=False,
        color=col,
        plot_centroids=[col],
        max_label_length=2,
        mask_obs=mask,
        sort_order=True,
        size=5,
        adjust_text_kwargs = {
            "arrowprops": {"arrowstyle": "-", "color": "black", "lw": 1},
            "expand_text": (1.05, 1.2),
            "expand_points": (2.0, 2.0), # ← expand bbox vs points; bigger = more gap from clusters
            "force_points": (1.5, 1.0), # ← repulsion strength from points; bigger = pushed further
        }
    )
    atl.pl.embedding(adata, basis=f'X_umap--{integration1}', **kwargs)
    atl.pl.embedding(adata, basis=f'X_umap--{integration2}', **kwargs)

In [ ]:
gd_colors = [f'{metric}:{comparison}'] # for comparison in comparisons]

kwargs = dict(
    color=gd_colors,
    cmap='viridis',
    size=5,
    ncols=1, # len(metrics),
    wspace=0.5,
    # vmin=0,
    vmax='p99.9',
    sort_order=True,
    mask_obs=mask,
)
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    **kwargs
)

sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration2}',
    **kwargs
)

## AT2 cells

In [ ]:
adata.obs['cluster'] = adata.obs[f'leiden_0.5_2--{integration1}']

In [ ]:
value_counts = adata.obs.query('ann_finest_level == "AT2"')['cluster'].value_counts()
adata.obs.loc[adata.obs['cluster'].isin(value_counts[value_counts < 10].index), 'cluster'] = float('nan')
adata.obs['cluster'] = adata.obs['cluster'].cat.remove_unused_categories()

In [ ]:
adata.obs['cluster'].value_counts()

In [ ]:
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    color='cluster',
    mask_obs=adata.obs['ann_finest_level'] == 'AT2',
    ncols=1,
    size=5,
    palette=sc.pl.palettes.default_102,
)

sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration2}',
    color='cluster',
    mask_obs=adata.obs['ann_finest_level'] == 'AT2',
    ncols=1,
    size=5,
    palette=sc.pl.palettes.default_102,
)

# Cluster-level analysis

In [ ]:
cluster_key = f'leiden_0.5_2--{integration1}'

cluster_high = plot_diff_per_clusters(
    adata,
    cluster_key,
    f'{metric}:{comparisons[1]}',
    # cluster_thresh=10,
    quantile=0.999,
    # agg='mean',
    width=5,
    height_factor=0.3,
    cluster_thresh=50,
)

In [ ]:
adata.obs['cluster_high'] = adata.obs.loc[
    adata.obs[cluster_key].isin(
        cluster_high.sort_values(ascending=False).head(10).index
    ),
    cluster_key
].cat.remove_unused_categories()

In [ ]:
kwargs = dict(
    plot_centroids=['cluster_high'],
    verbose=False,
    size=10,
    adjust_text_kwargs = {
        "arrowprops": {"arrowstyle": "-", "color": "black", "lw": 1},
        "expand_text": (1.05, 1.2),
        "expand_points": (2.0, 2.0), # ← expand bbox vs points; bigger = more gap from clusters
        "force_points": (1.0, 0.5), # ← repulsion strength from points; bigger = pushed further
    }
)

atl.pl.embedding(adata, basis=f'X_umap--{integration1}', **kwargs)
atl.pl.embedding(adata, basis=f'X_umap--{integration2}', **kwargs)

In [ ]:
plot_diff_per_clusters(
    adata,
    'cluster_high',
    f'{metric}:{comparisons[1]}',
    # cluster_thresh=10,
    quantile=0.999,
    # agg='mean',
    width=5,
    height_factor=0.3,
    cluster_thresh=50,
)